# Giai đoạn 5 — Đánh giá Test và chất lượng xác suất

Theo kế hoạch dự án v3.1. Nhãn `recommended_career` mô tả **nhóm nghề nghiệp được khuyến nghị**, không phải tỷ lệ trúng tuyển đại học.

In [ ]:
from pathlib import Path
import os, sys
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)
print("Project:", ROOT)

## 1. Xem chỉ số Test của mô hình *đã chọn từ Train CV*

In [ ]:
import json, pandas as pd
from career_ml.config import OUTPUT
metrics=json.loads((OUTPUT / "tables" / "test_metrics.json").read_text(encoding="utf-8"))
print('Test Metrics (từ lần đánh giá duy nhất sau lựa chọn):')
print(json.dumps(metrics,indent=2))
display(pd.read_csv(OUTPUT / "tables" / "classification_report.csv"))

## 2. Kiểm tra thứ tự vector xác suất, tổng bằng 1

In [ ]:
import joblib, numpy as np
from career_ml.data import load_data,prepare_xy,split_data
bundle=joblib.load(OUTPUT / "models" / "career_classifier.joblib")
X,y,_=prepare_xy(load_data(),excluded=bundle["excluded_columns"])
_,X_test,_,y_test=split_data(X,y)
model=bundle["model"]
proba=model.predict_proba(X_test.iloc[:5])
labels=bundle["label_encoder"].inverse_transform(model.classes_.astype(int))
print('Thứ tự các lớp:',list(labels))
print('Vector mẫu đầu tiên:');print(pd.Series(proba[0],index=labels))
assert (proba>=0).all() and (proba<=1).all()
assert np.allclose(proba.sum(axis=1),1)
print('PASS - xác suất có thứ tự phù hợp nhãn và tổng bằng 1')

## 3. Confusion Matrix và calibration từng lớp (one-vs-rest)

In [ ]:
from IPython.display import Image,display
for figure in ["confusion_matrix.png","calibration_12_classes.png"]:
    display(Image(filename=str(OUTPUT / "figures" / figure),width=900))
display(pd.read_csv(OUTPUT / "tables" / "calibration_bins.csv").head(20))

**Cần nhận xét trung thực:** Độ chính xác không đồng nghĩa xác suất được hiệu chỉnh. Log loss, reliability bin và Macro F1 đều phải xem xét. Không chọn lại calibration dựa trên Test.